# Panoptic Indoor Segmentation v5

## YOLO26 vs Mask R-CNN vs Mask2Former

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "smoke_tests":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "viskom_pipeline.py").exists():
    raise RuntimeError("Open this notebook from the Week11-13 project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viskom_pipeline import (
    YoloPolygonDataset,
    audit_dataset,
    available_completed_runs,
    environment_summary,
    evaluate_saved_experiment,
    experiment_matrix,
    train_experiment,
    visualize_ground_truth,
)

print(f"Project root: {PROJECT_ROOT}")

## 1. Verify CUDA and package versions

Save this output as part of the reproducibility evidence.

In [ ]:
ENVIRONMENT = environment_summary()
display(pd.DataFrame([ENVIRONMENT]).T.rename(columns={0: "value"}))
assert ENVIRONMENT["cuda_available"], "CUDA is unavailable in the selected kernel."
assert ENVIRONMENT["ultralytics"], "Install requirements.txt in this kernel."
assert ENVIRONMENT["transformers"], "Install requirements.txt in this kernel."
assert ENVIRONMENT["torchmetrics"], "Install requirements.txt in this kernel."

## 2. Locked experiment configuration

For the final experiment, do not change one member of a pretrained/scratch pair without making the identical change to its partner. To avoid accidental execution, ACTIVE_RUN_IDS starts empty. Add **one run ID at a time** after smoke tests pass.

In [ ]:
ORIGINAL_DATASET_ROOT = PROJECT_ROOT / "Roboflow_Dataset"
CUSTOM_DATASET_ROOT = PROJECT_ROOT / "My_Dataset"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "full_runs"
RESULTS_CSV = PROJECT_ROOT / "results.csv"

SEED = 42
INPUT_SIZE = 640
# EPOCHS = 30
EVAL_EVERY = 5

EPOCHS = 45

# ACTIVE_RUN_IDS = [
#     "yolo26n_seg_pretrained",
#     "yolo26n_seg_scratch",
#     "maskrcnn_r50_fpn_v2_pretrained",
#     "maskrcnn_r50_fpn_v2_scratch",
#     "mask2former_swin_tiny_pretrained",
#     "mask2former_swin_tiny_scratch",
# ]

# Recommended: put exactly one run ID here, run the training cell, then clear it.
ACTIVE_RUN_IDS = []
# ACTIVE_RUN_IDS = ["yolo26n_seg_pretrained"]
# ACTIVE_RUN_IDS = ["yolo26n_seg_scratch"]
# ACTIVE_RUN_IDS = ["maskrcnn_r50_fpn_v2_pretrained"]
# ACTIVE_RUN_IDS = ["mask2former_swin_tiny_pretrained"]
# ACTIVE_RUN_IDS = ["mask2former_swin_tiny_scratch"]

RUN_MATRIX = pd.DataFrame(experiment_matrix())
display(RUN_MATRIX)

## 3. Audit the original dataset

This verifies split sizes, polygon structure, class counts, malformed labels, and classes that occur in the test set without any training examples.

In [ ]:
original_audit = audit_dataset(ORIGINAL_DATASET_ROOT)
split_rows = []
for split, values in original_audit["splits"].items():
    split_rows.append({
        "split": split,
        "images": values["images"],
        "label_files": values["label_files"],
        "instances": values["instances"],
        "empty_label_files": values["empty_label_files"],
        "malformed_annotations": len(values["malformed_annotations"]),
    })
display(pd.DataFrame(split_rows))
print("Classes missing from train but present in test:")
print(original_audit["classes_missing_from_train_but_in_test"])
print("Classes with 1–9 training instances:")
print(original_audit["classes_with_fewer_than_10_train_instances"])

assert original_audit["num_classes"] == 74
assert not any(
    values["malformed_annotations"]
    for values in original_audit["splits"].values()
), "Fix malformed annotations before training."

### Visual polygon-decoding check

Change SAMPLE_INDEX and inspect several images from train, validation, and test. Masks should align with objects and class counts should be plausible.

In [ ]:
SAMPLE_SPLIT = "train"
SAMPLE_INDEX = 0
sample_dataset = YoloPolygonDataset(ORIGINAL_DATASET_ROOT, SAMPLE_SPLIT)
visualize_ground_truth(sample_dataset, SAMPLE_INDEX)

## 4. Audit the personal test set when annotation is complete

The cell safely skips while My_Dataset/data.yaml is absent. The personal set must remain test-only and must not enter training or validation.

In [ ]:
CUSTOM_READY = (CUSTOM_DATASET_ROOT / "data.yaml").exists()
if CUSTOM_READY:
    custom_audit = audit_dataset(CUSTOM_DATASET_ROOT)
    display(pd.DataFrame([
        {"split": split, **{k: v for k, v in values.items() if k in {"images", "label_files", "instances", "empty_label_files"}}}
        for split, values in custom_audit["splits"].items()
    ]))
    assert custom_audit["num_classes"] == 74
    assert custom_audit["splits"]["test"]["images"] >= 40
else:
    print("Personal test export is not ready yet. Continue annotation; full training is not blocked.")

## 5. Train selected full experiments

Each run saves native logs, checkpoints, timing, peak CUDA memory, and run_metadata.json under artifacts/full_runs/<run_id>/. Existing completed runs are protected unless OVERWRITE=True is explicitly set.

Training all six runs in one execution is discouraged. Run and verify one experiment at a time.

In [ ]:
OVERWRITE = False
if not ACTIVE_RUN_IDS:
    print("No active run selected. Set ACTIVE_RUN_IDS to one run ID when ready.")
for run_id in ACTIVE_RUN_IDS:
    print(f"Starting full experiment: {run_id}")
    metadata = train_experiment(
        run_id=run_id,
        dataset_root=ORIGINAL_DATASET_ROOT,
        artifact_root=ARTIFACT_ROOT,
        epochs=EPOCHS,
        input_size=INPUT_SIZE,
        seed=SEED,
        eval_every=EVAL_EVERY,
        overwrite=OVERWRITE,
    )
    display(pd.DataFrame([metadata]).T)

In [ ]:
COMPLETED_RUN_IDS = available_completed_runs(ARTIFACT_ROOT)
print(f"Completed runs ({len(COMPLETED_RUN_IDS)}/6):")
for run_id in COMPLETED_RUN_IDS:
    print(" -", run_id)

## 6. Evaluate completed runs on the original test set

All three frameworks are converted to the same TorchMetrics COCO-style segmentation mAP implementation. Re-running an evaluation replaces the matching run/evaluation row instead of creating a duplicate.

In [ ]:
EVALUATE_RUN_IDS = available_completed_runs(ARTIFACT_ROOT)
original_rows = []
for run_id in EVALUATE_RUN_IDS:
    print(f"Evaluating {run_id} on original_test...")
    row = evaluate_saved_experiment(
        run_id=run_id,
        artifact_root=ARTIFACT_ROOT,
        evaluation_root=ORIGINAL_DATASET_ROOT,
        evaluation_split="test",
        evaluation_name="original_test",
        results_csv=RESULTS_CSV,
    )
    original_rows.append(row)
display(pd.DataFrame(original_rows)) if original_rows else print("No completed runs to evaluate.")

## 7. Evaluate completed runs on the personal test set

Run only after all personal annotations are finished, quality-checked, and exported to My_Dataset/.

In [ ]:
custom_rows = []
if not (CUSTOM_DATASET_ROOT / "data.yaml").exists():
    print("Personal test set is not exported yet; custom evaluation skipped.")
else:
    for run_id in available_completed_runs(ARTIFACT_ROOT):
        print(f"Evaluating {run_id} on personal_test...")
        row = evaluate_saved_experiment(
            run_id=run_id,
            artifact_root=ARTIFACT_ROOT,
            evaluation_root=CUSTOM_DATASET_ROOT,
            evaluation_split="test",
            evaluation_name="personal_test",
            results_csv=RESULTS_CSV,
        )
        custom_rows.append(row)
display(pd.DataFrame(custom_rows)) if custom_rows else None

## 8. Consolidated results and domain shift

The final CSV should contain 12 rows: six runs evaluated on two test sets. Domain shift is calculated as personal_test minus original_test; negative values indicate degradation on the phone-photo dataset.

In [ ]:
results = pd.read_csv(RESULTS_CSV)
display(results)

if len(results) > 0 and {"original_test", "personal_test"}.issubset(set(results["evaluation_dataset"])):
    comparison = results.pivot(
        index=["run_id", "model_name", "initialization"],
        columns="evaluation_dataset",
        values="mask_map_50_95",
    ).reset_index()
    comparison["domain_shift_delta"] = comparison["personal_test"] - comparison["original_test"]
    display(comparison.sort_values("domain_shift_delta"))
else:
    print("Domain-shift table requires both evaluation sets.")